# 🚇 FIAP — Dynamic Programming | Checkpoint 2
## Memoização + Rotas de Metrô


---

Grupo
Rafael Falaguasta Ferraz RM561714./
Victor Simoes Altieri RM565288 /
Eduardo Antonio Delarissia RM563468 /

In [1]:
import subprocess, sys

try:
    from google.colab import drive
    IN_COLAB = True
    print('✓ Rodando no Google Colab!')
except ImportError:
    IN_COLAB = False
    print('✓ Rodando localmente')

# Instalar dependências
print('\n📦 Instalando bibliotecas...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'folium', 'pandas'])
print('✓ Pronto!')


✓ Rodando no Google Colab!

📦 Instalando bibliotecas...
✓ Pronto!


In [2]:
import time, tracemalloc, functools
from typing import Dict, Tuple, List, Set, FrozenSet
import pandas as pd
import folium
import sys

print('✓ Todas as bibliotecas importadas!')
print(f'🐍 Python: {sys.version.split()[0]}')
print(f'📊 Pandas: {pd.__version__}')
print(f'🗺️  Folium: {folium.__version__}')


✓ Todas as bibliotecas importadas!
🐍 Python: 3.12.13
📊 Pandas: 2.2.2
🗺️  Folium: 0.20.0


# [2] GRAFOS - 3 CIDADES


In [3]:
# Beijing Metro
grafo_beijing = {
    'Sihui East': [('Sihui', 2), ('Guomao', 3)],
    'Sihui': [('Sihui East', 2), ('Huanghuayuan', 2)],
    'Huanghuayuan': [('Sihui', 2), ('Pingguoyuan', 2)],
    'Pingguoyuan': [('Huanghuayuan', 2), ('Gaoguanzhuang', 3)],
    'Guomao': [('Sihui East', 3), ('Chaoyang', 2), ('Dawanglu', 2)],
    'Chaoyang': [('Guomao', 2), ('Chaoyangmen', 2)],
    'Dawanglu': [('Guomao', 2), ('Jianguo', 2)],
    'Chaoyangmen': [('Chaoyang', 2), ('Jianguomen', 2), ('Xuanwumen', 3)],
    'Jianguomen': [('Chaoyangmen', 2), ('Hepingmen', 2)],
    'Hepingmen': [('Jianguomen', 2), ('Caishikou', 2)],
    'Caishikou': [('Hepingmen', 2), ('Xuanwumen', 2)],
    'Xuanwumen': [('Caishikou', 2), ('Chaoyangmen', 3), ('Lingjing Hutong', 2)],
    'Lingjing Hutong': [('Xuanwumen', 2), ('Xizhimen', 2)],
    'Xizhimen': [('Lingjing Hutong', 2), ('Changchunjie', 2)],
    'Changchunjie': [('Xizhimen', 2), ('Fuxingmen', 2)],
    'Fuxingmen': [('Changchunjie', 2), ('Xuanwumen', 3)],
    'Jianguo': [('Dawanglu', 2), ('Gaoguanzhuang', 2)],
    'Gaoguanzhuang': [('Pingguoyuan', 3), ('Jianguo', 2)],
}

coords_beijing = {
    'Sihui East': (39.9134, 116.4706),
    'Sihui': (39.9134, 116.4656),
    'Huanghuayuan': (39.9134, 116.4556),
    'Pingguoyuan': (39.9134, 116.4456),
    'Guomao': (39.9067, 116.4626),
    'Chaoyang': (39.9084, 116.4683),
    'Dawanglu': (39.9050, 116.4593),
    'Chaoyangmen': (39.9176, 116.4276),
    'Jianguomen': (39.9143, 116.4176),
    'Hepingmen': (39.9093, 116.4076),
    'Caishikou': (39.8993, 116.4076),
    'Xuanwumen': (39.9026, 116.3976),
    'Lingjing Hutong': (39.9093, 116.3876),
    'Xizhimen': (39.9176, 116.3776),
    'Changchunjie': (39.9226, 116.3776),
    'Fuxingmen': (39.9276, 116.3876),
    'Jianguo': (39.8950, 116.4593),
    'Gaoguanzhuang': (39.8850, 116.4593),
}

print(f'✓ Beijing: {len(grafo_beijing)} estações')


✓ Beijing: 18 estações


In [4]:
# San Francisco BART
grafo_sf = {
    'Dublin/Pleasanton': [('West Dublin', 3), ('Pleasanton', 2)],
    'Pleasanton': [('Dublin/Pleasanton', 2), ('Livermore', 3)],
    'Livermore': [('Pleasanton', 3), ('Livermore Center', 2)],
    'Livermore Center': [('Livermore', 2), ('South Livermore', 2)],
    'South Livermore': [('Livermore Center', 2), ('West Dublin', 4)],
    'West Dublin': [('Dublin/Pleasanton', 3), ('South Livermore', 4), ('Union City', 4)],
    'Union City': [('West Dublin', 4), ('South Hayward', 2), ('Hayward', 2)],
    'South Hayward': [('Union City', 2), ('Hayward', 2)],
    'Hayward': [('Union City', 2), ('South Hayward', 2), ('Bay Fair', 3)],
    'Bay Fair': [('Hayward', 3), ('Coliseum', 2)],
    'Coliseum': [('Bay Fair', 2), ('Fruitvale', 2)],
    'Fruitvale': [('Coliseum', 2), ('Lake Merritt', 2)],
    'Lake Merritt': [('Fruitvale', 2), ('19th St', 2), ('Embarcadero', 3)],
    '19th St': [('Lake Merritt', 2), ('12th St', 2)],
    '12th St': [('19th St', 2), ('West Oakland', 2)],
    'West Oakland': [('12th St', 2), ('Embarcadero', 3)],
    'Embarcadero': [('Lake Merritt', 3), ('West Oakland', 3), ('Montgomery St', 2)],
    'Montgomery St': [('Embarcadero', 2), ('Powell St', 2)],
    'Powell St': [('Montgomery St', 2), ('Civic Center', 2)],
    'Civic Center': [('Powell St', 2), ('Van Ness', 2)],
    'Van Ness': [('Civic Center', 2), ('Daly City', 3)],
    'Daly City': [('Van Ness', 3), ('Colma', 2)],
    'Colma': [('Daly City', 2)],
}

coords_sf = {
    'Dublin/Pleasanton': (37.7015, -121.8853),
    'Pleasanton': (37.7280, -121.8747),
    'Livermore': (37.6821, -121.7685),
    'Livermore Center': (37.6740, -121.7601),
    'South Livermore': (37.6650, -121.7515),
    'West Dublin': (37.7100, -121.8700),
    'Union City': (37.5915, -122.0171),
    'South Hayward': (37.6348, -122.0814),
    'Hayward': (37.6689, -122.0808),
    'Bay Fair': (37.7168, -122.1408),
    'Coliseum': (37.7543, -122.1969),
    'Fruitvale': (37.7754, -122.2241),
    'Lake Merritt': (37.7976, -122.2713),
    '19th St': (37.8096, -122.2717),
    '12th St': (37.8284, -122.2717),
    'West Oakland': (37.8045, -122.2948),
    'Embarcadero': (37.7930, -122.3964),
    'Montgomery St': (37.7896, -122.4012),
    'Powell St': (37.7849, -122.4085),
    'Civic Center': (37.7797, -122.4136),
    'Van Ness': (37.7750, -122.4200),
    'Daly City': (37.7061, -122.4696),
    'Colma': (37.6847, -122.4645),
}

print(f'✓ San Francisco: {len(grafo_sf)} estações')


✓ San Francisco: 23 estações


In [5]:
# São Paulo Metro + CPTM
grafo_sp = {
    'Tucuruvi': [('Parada Inglesa', 2), ('Jardim São Paulo', 3)],
    'Parada Inglesa': [('Tucuruvi', 2), ('Santana', 2)],
    'Santana': [('Parada Inglesa', 2), ('Carandiru', 2)],
    'Carandiru': [('Santana', 2), ('Tiradentes', 2)],
    'Tiradentes': [('Carandiru', 2), ('Luz', 2)],
    'Luz': [('Tiradentes', 2), ('República', 2)],
    'República': [('Luz', 2), ('Sé', 2)],
    'Sé': [('República', 2), ('Liberdade', 2), ('Brooklin', 3)],
    'Liberdade': [('Sé', 2), ('Paraíso', 2)],
    'Paraíso': [('Liberdade', 2), ('Vergueiro', 2)],
    'Vergueiro': [('Paraíso', 2), ('Estação da Luz', 3)],
    'Jardim São Paulo': [('Tucuruvi', 3), ('Parada Inglesa', 3)],
    'Capão Redondo': [('Campo Limpo', 2), ('Vila das Belezas', 2)],
    'Campo Limpo': [('Capão Redondo', 2), ('Vila Madalena', 2)],
    'Vila Madalena': [('Campo Limpo', 2), ('Largo da Batata', 2)],
    'Largo da Batata': [('Vila Madalena', 2), ('Faria Lima', 2)],
    'Faria Lima': [('Largo da Batata', 2), ('Clínicas', 2)],
    'Clínicas': [('Faria Lima', 2), ('Brooklin', 2)],
    'Brooklin': [('Clínicas', 2), ('Sé', 3)],
    'Vila das Belezas': [('Capão Redondo', 2), ('Campo Limpo', 2)],
    'Estação da Luz': [('Vergueiro', 3), ('Liberdade', 3)],
}

coords_sp = {
    'Tucuruvi': (-23.5026, -46.6004),
    'Parada Inglesa': (-23.5098, -46.5952),
    'Santana': (-23.5150, -46.5890),
    'Carandiru': (-23.5206, -46.5840),
    'Tiradentes': (-23.5261, -46.5790),
    'Luz': (-23.5310, -46.5740),
    'República': (-23.5380, -46.5690),
    'Sé': (-23.5505, -46.6338),
    'Liberdade': (-23.5570, -46.6404),
    'Paraíso': (-23.5620, -46.6454),
    'Vergueiro': (-23.5670, -46.6504),
    'Jardim São Paulo': (-23.4980, -46.5954),
    'Capão Redondo': (-23.6470, -46.7540),
    'Campo Limpo': (-23.6380, -46.7404),
    'Vila Madalena': (-23.6270, -46.7240),
    'Largo da Batata': (-23.6150, -46.7070),
    'Faria Lima': (-23.6040, -46.6880),
    'Clínicas': (-23.5920, -46.6660),
    'Brooklin': (-23.5590, -46.6520),
    'Vila das Belezas': (-23.6520, -46.7420),
    'Estação da Luz': (-23.5400, -46.5640),
}

print(f'✓ São Paulo: {len(grafo_sp)} estações')


✓ São Paulo: 21 estações


# [3] FATORES DE HORÁRIO


In [6]:
def fator_horario(hora: int) -> float:
    if 5 <= hora < 7:
        return 0.6
    elif 7 <= hora < 9:
        return 1.5
    elif 9 <= hora < 17:
        return 1.0
    elif 17 <= hora < 20:
        return 2.0
    else:
        return 0.8

print('\n📊 TABELA DE FATORES:')
print('| Horário | Fator |')
print('|---------|-------|')
print(f'| 5h-7h   | {fator_horario(6):.1f}  |')
print(f'| 7h-9h   | {fator_horario(8):.1f}  |')
print(f'| 9h-17h  | {fator_horario(12):.1f}  |')
print(f'| 17h-20h | {fator_horario(18):.1f}  |')
print(f'| 20h+    | {fator_horario(22):.1f}  |')



📊 TABELA DE FATORES:
| Horário | Fator |
|---------|-------|
| 5h-7h   | 0.6  |
| 7h-9h   | 1.5  |
| 9h-17h  | 1.0  |
| 17h-20h | 2.0  |
| 20h+    | 0.8  |


# [4] ALGORITMOS


In [7]:
class CaminhoMaisCurto:
    def __init__(self, grafo, fator_func):
        self.grafo = grafo
        self.fator_func = fator_func
        self.cache = {}
        self.chamadas = 0

    def buscar(self, origem, destino, horario, visitados=frozenset()):
        self.chamadas += 1

        if origem == destino:
            return (0, [origem])

        chave = (origem, destino, horario, visitados)
        if chave in self.cache:
            return self.cache[chave]

        if origem not in self.grafo:
            return (float('inf'), [])

        melhor_custo = float('inf')
        melhor_caminho = []

        for vizinho, peso in self.grafo[origem]:
            if vizinho not in visitados:
                peso_ajustado = peso * self.fator_func(horario)
                custo_futuro, caminho_futuro = self.buscar(
                    vizinho, destino, horario, visitados | {origem}
                )
                custo_total = peso_ajustado + custo_futuro

                if custo_total < melhor_custo:
                    melhor_custo = custo_total
                    melhor_caminho = [origem] + caminho_futuro

        resultado = (melhor_custo, melhor_caminho)
        self.cache[chave] = resultado
        return resultado

print('✓ CaminhoMaisCurto implementado')


✓ CaminhoMaisCurto implementado


In [8]:
class CaminhoMaisLongo:
    def __init__(self, grafo, fator_func):
        self.grafo = grafo
        self.fator_func = fator_func
        self.chamadas = 0

    def buscar(self, origem, destino, horario, visitados=None):
        self.chamadas += 1

        if visitados is None:
            visitados = set()

        if origem == destino:
            return (0, [origem])

        if origem not in self.grafo:
            return (float('-inf'), [])

        visitados.add(origem)
        pior_custo = float('-inf')
        pior_caminho = []

        for vizinho, peso in self.grafo[origem]:
            if vizinho not in visitados:
                peso_ajustado = peso * self.fator_func(horario)
                custo_futuro, caminho_futuro = self.buscar(
                    vizinho, destino, horario, visitados.copy()
                )

                if caminho_futuro:
                    custo_total = peso_ajustado + custo_futuro
                    if custo_total > pior_custo:
                        pior_custo = custo_total
                        pior_caminho = [origem] + caminho_futuro

        return (pior_custo, pior_caminho)

print('✓ CaminhoMaisLongo implementado')


✓ CaminhoMaisLongo implementado


# [5] EXECUÇÃO - BEIJING


In [9]:
print('\n' + '='*70)
print('🇨🇳 BEIJING')
print('='*70)

origem_bj = 'Sihui East'
destino_bj = 'Xizhimen'
horario_bj = 18

print(f'\n📍 {origem_bj} → {destino_bj}')
print(f'🕐 {horario_bj}h (fator {fator_horario(horario_bj):.1f}x)')

bj_curto = CaminhoMaisCurto(grafo_beijing, fator_horario)
tracemalloc.start()
t0 = time.perf_counter()
custo_bj, caminho_bj = bj_curto.buscar(origem_bj, destino_bj, horario_bj)
t1 = time.perf_counter()
mem_atual, mem_pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'\n💰 Custo: {custo_bj:.2f} min')
print(f'🛣️  Caminho: {" → ".join(caminho_bj)}')
print(f'⏱️  Tempo: {(t1-t0)*1000:.4f} ms')
print(f'💾 Memória: {mem_pico/1024:.2f} KB')
print(f'📞 Chamadas: {bj_curto.chamadas}')

bj_longo = CaminhoMaisLongo(grafo_beijing, fator_horario)
t0 = time.perf_counter()
custo_longo_bj, caminho_longo_bj = bj_longo.buscar(origem_bj, destino_bj, horario_bj)
t1 = time.perf_counter()

print(f'\n🔝 CAMINHO LONGO:')
print(f'💰 Custo: {custo_longo_bj:.2f} min')
print(f'🛣️  {" → ".join(caminho_longo_bj)}')
print(f'🚉 Estações: {len(caminho_longo_bj)}')



🇨🇳 BEIJING

📍 Sihui East → Xizhimen
🕐 18h (fator 2.0x)

💰 Custo: 28.00 min
🛣️  Caminho: Sihui East → Guomao → Chaoyang → Chaoyangmen → Xuanwumen → Lingjing Hutong → Xizhimen
⏱️  Tempo: 2.1105 ms
💾 Memória: 43.49 KB
📞 Chamadas: 43

🔝 CAMINHO LONGO:
💰 Custo: 62.00 min
🛣️  Sihui East → Sihui → Huanghuayuan → Pingguoyuan → Gaoguanzhuang → Jianguo → Dawanglu → Guomao → Chaoyang → Chaoyangmen → Jianguomen → Hepingmen → Caishikou → Xuanwumen → Lingjing Hutong → Xizhimen
🚉 Estações: 16


# [6] EXECUÇÃO - SAN FRANCISCO


In [10]:
print('\n' + '='*70)
print('🇺🇸 SAN FRANCISCO')
print('='*70)

origem_sf = 'Dublin/Pleasanton'
destino_sf = 'Daly City'
horario_sf = 18

print(f'\n📍 {origem_sf} → {destino_sf}')
print(f'🕐 {horario_sf}h (fator {fator_horario(horario_sf):.1f}x)')

sf_curto = CaminhoMaisCurto(grafo_sf, fator_horario)
tracemalloc.start()
t0 = time.perf_counter()
custo_sf, caminho_sf = sf_curto.buscar(origem_sf, destino_sf, horario_sf)
t1 = time.perf_counter()
mem_atual, mem_pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'\n💰 Custo: {custo_sf:.2f} min')
print(f'🛣️  Caminho: {" → ".join(caminho_sf)}')
print(f'⏱️  Tempo: {(t1-t0)*1000:.4f} ms')
print(f'💾 Memória: {mem_pico/1024:.2f} KB')
print(f'📞 Chamadas: {sf_curto.chamadas}')

sf_longo = CaminhoMaisLongo(grafo_sf, fator_horario)
t0 = time.perf_counter()
custo_longo_sf, caminho_longo_sf = sf_longo.buscar(origem_sf, destino_sf, horario_sf)
t1 = time.perf_counter()

print(f'\n🔝 CAMINHO LONGO:')
print(f'💰 Custo: {custo_longo_sf:.2f} min')
print(f'🛣️  {" → ".join(caminho_longo_sf)}')
print(f'🚉 Estações: {len(caminho_longo_sf)}')



🇺🇸 SAN FRANCISCO

📍 Dublin/Pleasanton → Daly City
🕐 18h (fator 2.0x)

💰 Custo: 64.00 min
🛣️  Caminho: Dublin/Pleasanton → West Dublin → Union City → Hayward → Bay Fair → Coliseum → Fruitvale → Lake Merritt → Embarcadero → Montgomery St → Powell St → Civic Center → Van Ness → Daly City
⏱️  Tempo: 4.2558 ms
💾 Memória: 106.81 KB
📞 Chamadas: 109

🔝 CAMINHO LONGO:
💰 Custo: 100.00 min
🛣️  Dublin/Pleasanton → Pleasanton → Livermore → Livermore Center → South Livermore → West Dublin → Union City → South Hayward → Hayward → Bay Fair → Coliseum → Fruitvale → Lake Merritt → 19th St → 12th St → West Oakland → Embarcadero → Montgomery St → Powell St → Civic Center → Van Ness → Daly City
🚉 Estações: 22


# [7] EXECUÇÃO - SÃO PAULO


In [11]:
print('\n' + '='*70)
print('🇧🇷 SÃO PAULO')
print('='*70)

origem_sp = 'Tucuruvi'
destino_sp = 'Capão Redondo'
horario_sp = 18

print(f'\n📍 {origem_sp} (Linha 1) → {destino_sp} (Linha 5)')
print(f'🕐 {horario_sp}h (fator {fator_horario(horario_sp):.1f}x)')
print(f'🔗 Integração: Sé e Brooklin')

sp_curto = CaminhoMaisCurto(grafo_sp, fator_horario)
tracemalloc.start()
t0 = time.perf_counter()
custo_sp, caminho_sp = sp_curto.buscar(origem_sp, destino_sp, horario_sp)
t1 = time.perf_counter()
mem_atual, mem_pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'\n💰 Custo: {custo_sp:.2f} min')
print(f'🛣️  Caminho: {" → ".join(caminho_sp)}')
print(f'⏱️  Tempo: {(t1-t0)*1000:.4f} ms')
print(f'💾 Memória: {mem_pico/1024:.2f} KB')
print(f'📞 Chamadas: {sp_curto.chamadas}')

sp_longo = CaminhoMaisLongo(grafo_sp, fator_horario)
t0 = time.perf_counter()
custo_longo_sp, caminho_longo_sp = sp_longo.buscar(origem_sp, destino_sp, horario_sp)
t1 = time.perf_counter()

print(f'\n🔝 CAMINHO LONGO:')
print(f'💰 Custo: {custo_longo_sp:.2f} min')
print(f'🛣️  {" → ".join(caminho_longo_sp)}')
print(f'🚉 Estações: {len(caminho_longo_sp)}')



🇧🇷 SÃO PAULO

📍 Tucuruvi (Linha 1) → Capão Redondo (Linha 5)
🕐 18h (fator 2.0x)
🔗 Integração: Sé e Brooklin

💰 Custo: 58.00 min
🛣️  Caminho: Tucuruvi → Parada Inglesa → Santana → Carandiru → Tiradentes → Luz → República → Sé → Brooklin → Clínicas → Faria Lima → Largo da Batata → Vila Madalena → Campo Limpo → Capão Redondo
⏱️  Tempo: 1.4620 ms
💾 Memória: 45.44 KB
📞 Chamadas: 38

🔝 CAMINHO LONGO:
💰 Custo: 66.00 min
🛣️  Tucuruvi → Jardim São Paulo → Parada Inglesa → Santana → Carandiru → Tiradentes → Luz → República → Sé → Brooklin → Clínicas → Faria Lima → Largo da Batata → Vila Madalena → Campo Limpo → Capão Redondo
🚉 Estações: 16


# [8] MAPAS FOLIUM


In [12]:
# Mapa Beijing
mapa_bj = folium.Map(location=(39.9042, 116.4074), zoom_start=12)

for estacao, (lat, lng) in coords_beijing.items():
    if estacao == origem_bj:
        cor = 'green'
    elif estacao == destino_bj:
        cor = 'red'
    else:
        cor = 'blue'

    folium.CircleMarker((lat, lng), radius=6, popup=estacao, color=cor, fill=True, fillColor=cor).add_to(mapa_bj)

if len(caminho_bj) > 1:
    coords = [coords_beijing[e] for e in caminho_bj]
    folium.PolyLine(coords, color='orange', weight=4, opacity=0.9).add_to(mapa_bj)

if len(caminho_longo_bj) > 1:
    coords = [coords_beijing[e] for e in caminho_longo_bj]
    folium.PolyLine(coords, color='purple', weight=3, opacity=0.6, dash_array='5,5').add_to(mapa_bj)

print('✓ Mapa Beijing criado')
mapa_bj


✓ Mapa Beijing criado


In [13]:
# Mapa San Francisco
mapa_sf = folium.Map(location=(37.7749, -122.4194), zoom_start=11)

for estacao, (lat, lng) in coords_sf.items():
    cor = 'green' if estacao == origem_sf else 'red' if estacao == destino_sf else 'blue'
    folium.CircleMarker((lat, lng), radius=6, popup=estacao, color=cor, fill=True, fillColor=cor).add_to(mapa_sf)

if len(caminho_sf) > 1:
    coords = [coords_sf[e] for e in caminho_sf]
    folium.PolyLine(coords, color='orange', weight=4, opacity=0.9).add_to(mapa_sf)

if len(caminho_longo_sf) > 1:
    coords = [coords_sf[e] for e in caminho_longo_sf]
    folium.PolyLine(coords, color='purple', weight=3, opacity=0.6, dash_array='5,5').add_to(mapa_sf)

print('✓ Mapa SF criado')
mapa_sf


✓ Mapa SF criado


In [14]:
# Mapa São Paulo
mapa_sp = folium.Map(location=(-23.5505, -46.6333), zoom_start=12)

for estacao, (lat, lng) in coords_sp.items():
    if estacao == origem_sp:
        cor = 'green'
    elif estacao == destino_sp:
        cor = 'red'
    elif estacao in ['Sé', 'Brooklin']:
        cor = 'gold'
    else:
        cor = 'blue'
    folium.CircleMarker((lat, lng), radius=6, popup=estacao, color=cor, fill=True, fillColor=cor).add_to(mapa_sp)

if len(caminho_sp) > 1:
    coords = [coords_sp[e] for e in caminho_sp]
    folium.PolyLine(coords, color='orange', weight=4, opacity=0.9).add_to(mapa_sp)

if len(caminho_longo_sp) > 1:
    coords = [coords_sp[e] for e in caminho_longo_sp]
    folium.PolyLine(coords, color='purple', weight=3, opacity=0.6, dash_array='5,5').add_to(mapa_sp)

print('✓ Mapa SP criado')
mapa_sp


✓ Mapa SP criado


# [9] CONCLUSÕES

## Complexidade Big-O

**Sem Memoização:** O(V^E) — EXPONENCIAL

**Com Memoização:** O(V² + E) — POLINOMIAL
